# Installation/Setup

In [1]:
import os
import sys
import subprocess
import copy
import yaml
import pandas as pd
import numpy as np
import plotly.io as pio
from IPython.display import display, Image, SVG
from ipywidgets import interact, Dropdown
from ipymolstar import PDBeMolstar

# Force a SINGLE Plotly renderer -- when multiple Jupyter/Plotly frontend extensions are
# active (common in VS Code), Plotly's auto-detection can set pio.renderers.default to a
# combined string like "vscode+notebook_connected", so every fig.show() call renders once
# per registered renderer, stacking visible duplicates of every single plot in the
# notebook. Pinning to one explicit renderer avoids that regardless of what got detected.
pio.renderers.default = 'vscode'

BECLUST3D_PATH = '/Users/ymyung/Projects/BEClust3D/src/beclust3d-public'
sys.path.insert(0, BECLUST3D_PATH)
sys.path.insert(0, os.path.join(BECLUST3D_PATH, 'examples'))

from be3d_local_helper import (
    show_svgs, show_images, plot_residue_dot, plot_ppi_vs_noppi_scatter,
    render_molstar, load_molstar_pdb, color_molstar, chain_values_from_df, edit_yaml_widgets,
)
from be3d_plotly import (
    show_side_by_side, show_stacked, show_picker, plot_hypothesis_qa, plot_violin_by_processed_muttype, plot_score_scatter,
    plot_dendrogram, plot_meta_dendrogram, plot_lfc_lfc3d_scatter, plot_plddt_rsa_scatter,
    plot_domain_barplot, plot_plddt_dis_barplot, plot_enrichment_test,
    plot_meta_score_scatter, plot_meta_lfc_lfc3d_scatter, plot_meta_plddt_rsa_scatter,
    plot_meta_domain_barplot, plot_meta_plddt_dis_barplot,
    COLOR_POS, COLOR_NEG,
)

# Assumes DSSP, ClustalO, and MUSCLE are already installed locally (see the public repo's
# README for install instructions) -- unlike the Colab notebook, this one never shells out
# to apt-get/wget for setup.

def run_be3d(yaml_path):
    script = os.path.join(BECLUST3D_PATH, 'examples', 'be3d_local.py')
    # Capture + print explicitly rather than letting the child inherit stdout/stderr --
    # a subprocess's inherited file descriptors don't reliably show up in a notebook
    # cell's own output (Jupyter/Colab capture sys.stdout at the Python level, which a
    # child process's raw fd can bypass), so check=True alone can raise CalledProcessError
    # with no visible clue about what actually went wrong inside be3d_local.py.
    result = subprocess.run([sys.executable, script, yaml_path], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        result.check_returncode()

def load_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)

def run_be3d_if_needed(yaml_path, done_marker):
    if os.path.exists(done_marker):
        print(f'[skip] {done_marker} already exists')
    else:
        run_be3d(yaml_path)


# Settings
- Check user intent: monomer, ppi mode, blind-target mode
- Show default settings
- As an example, we can use KBTBD4–HDAC1 case which supports monomer, ppi and ppi(blind) mode examples


In [2]:
# KBTBD4-HDAC1 (8VOJ) supports all three modes: monomer, ppi (via ppi_diff), and blind_target.
MONOMER_YAML = '/Users/ymyung/Projects/BEClust3D/be3d_test/KBTBD4_chain_B.yaml'
PPI_YAML = '/Users/ymyung/Projects/BEClust3D/be3d_test/ppi_diff_KBTBD4_HDAC1.yaml'
BLIND_TARGET_YAML = '/Users/ymyung/Projects/BEClust3D/be3d_test/blind_target_KBTBD4_HDAC1.yaml'

for label, path in [('monomer', MONOMER_YAML), ('ppi (ppi_diff)', PPI_YAML), ('blind_target', BLIND_TARGET_YAML)]:
    cfg = load_yaml(path)
    print(f"--- {label}: mode='{cfg.get('mode')}' ---")
    shown = {k: cfg[k] for k in ('input_gene', 'input_uniprot', 'input_chain', 'output_dir') if k in cfg}
    print(yaml.safe_dump(shown, sort_keys=False))


--- monomer: mode='monomer' ---
input_gene: KBTBD4
input_uniprot: Q9NVX7-2
input_chain: B
output_dir: /Users/ymyung/Projects/BEClust3D/be3d_test/output/KBTBD4_chain_B

--- ppi (ppi_diff): mode='ppi_diff' ---
input_gene: KBTBD4, HDAC1
input_uniprot: Q9NVX7-2, Q13547
input_chain: B, C
output_dir: /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1

--- blind_target: mode='blind_target' ---
input_gene: KBTBD4
input_uniprot: Q9NVX7-2
input_chain: B
output_dir: /Users/ymyung/Projects/BEClust3D/be3d_test/output/blind_target/KBTBD4_HDAC1



## Select a mode
- Pick one of monomer / ppi / blind_target below
- Only the section(s) matching the selected mode actually run further down; the others print a skip message

In [3]:
import ipywidgets as widgets

MODE_OPTIONS = [
    ('Monomer -- single target gene, no PPI partner', 'monomer'),
    ('PPI -- target gene(s) compared with vs. without a PPI partner (mode: ppi_diff)', 'ppi'),
    ('Blind target -- target has no screen data of its own, scored purely from PPI partner(s)', 'blind_target'),
]

mode_selector = widgets.RadioButtons(
    options=MODE_OPTIONS, description='Mode:',
    style={'description_width': '60px'}, layout=widgets.Layout(width='750px'),
)

MODE = mode_selector.value

def _on_mode_change(change):
    global MODE
    MODE = change['new']
    print(f"Selected mode: '{MODE}' -- re-run the config editor and the cells below for this mode.")

mode_selector.observe(_on_mode_change, names='value')
display(mode_selector)
print(f"Selected mode: '{MODE}' -- re-run the config editor and the cells below for this mode.")


RadioButtons(description='Mode:', layout=Layout(width='750px'), options=(('Monomer -- single target gene, no P…

Selected mode: 'monomer' -- re-run the config editor and the cells below for this mode.


## Edit config for the selected mode
- Adjust the selected mode's yaml settings before running the pipeline
- Each field is explained below it; changes save back to the yaml file immediately

In [5]:
YAML_BY_MODE = {'monomer': MONOMER_YAML, 'ppi': PPI_YAML, 'blind_target': BLIND_TARGET_YAML}
EDITABLE_KEYS_BY_MODE = {
    'monomer': ['input_gene', 'input_uniprot', 'input_chain', 'screen_dir', 'screens',
                'output_dir', 'user_pdb', 'user_fasta', 'user_dssp',
                'nRandom', 'structure_radius', 'clustering_radius',
                'function_for_lfc', 'function_for_lfc3d', 'function_for_meta'],
    'ppi': ['input_gene', 'input_uniprot', 'input_chain', 'screen_dir', 'screens',
            'output_dir', 'user_pdb', 'user_fasta', 'user_dssp', 'score_type', 'skip_existing',
            'nRandom', 'structure_radius', 'clustering_radius',
            'function_for_lfc', 'function_for_lfc3d', 'function_for_meta'],
    'blind_target': ['input_gene', 'input_uniprot', 'input_chain', 'output_dir',
                      'user_pdb', 'user_fasta', 'user_dssp',
                      'function_for_lfc', 'function_for_lfc3d', 'function_for_meta'],
}

print(f"Editing config for mode '{MODE}' ({YAML_BY_MODE[MODE]}) -- "
      "changes below are written back to the yaml file immediately, picked up the next "
      "time a cell further down runs the pipeline. Nested settings (pthr, database, "
      "mutation_category, qa, ...) aren't exposed here -- edit the yaml file directly for those.")
edit_yaml_widgets(YAML_BY_MODE[MODE], EDITABLE_KEYS_BY_MODE[MODE])


Editing config for mode 'ppi' (/Users/ymyung/Projects/BEClust3D/be3d_test/ppi_diff_KBTBD4_HDAC1.yaml) -- changes below are written back to the yaml file immediately, picked up the next time a cell further down runs the pipeline. Nested settings (pthr, database, mutation_category, qa, ...) aren't exposed here -- edit the yaml file directly for those.


# BE-QA
- QA plots (KS2)
- QA plot (violing plot)

In [6]:
if MODE == 'monomer':
    monomer_cfg = load_yaml(YAML_BY_MODE['monomer'])
    monomer_dir, monomer_gene, monomer_uniprot = monomer_cfg['output_dir'], monomer_cfg['input_gene'], monomer_cfg['input_uniprot']
    run_be3d_if_needed(YAML_BY_MODE['monomer'], os.path.join(monomer_dir, 'RUN_COMPLETED.txt'))

    monomer_screens = [s.strip().split('.')[0] for s in monomer_cfg['screens'].split(',')]

    monomer_hyp_ks = plot_hypothesis_qa(monomer_dir, test='KolmogorovSmirnov')
    monomer_hyp_mw = plot_hypothesis_qa(monomer_dir, test='MannWhitney')

    def show_monomer_qa(screen_name):
        print('QA (KS2, MW test, all screens) and processed LFC distribution by mutation category '
                '(violin, post mutation_priority + per-category filtering):')
        violin_fig = plot_violin_by_processed_muttype(monomer_dir, monomer_gene, screen_name)
        show_side_by_side(monomer_hyp_ks, monomer_hyp_mw, violin_fig, width=600, height=400, spacing=0.08)

    interact(show_monomer_qa, screen_name=Dropdown(options=monomer_screens, description='Screen:'));

elif MODE == 'ppi':
    ppi_cfg = load_yaml(YAML_BY_MODE['ppi'])
    ppi_root = ppi_cfg['output_dir']
    gene_names = [g.strip() for g in ppi_cfg['input_gene'].split(',')]
    ppi_screens = [s.strip().split('.')[0] for s in ppi_cfg['screens'].split(',')]

    # The QA/violin cells below need the per-gene no_ppi and ppi legs to already exist, so run
    # the pipeline first (mirrors what the BE-Clust3D (PPI) cell does further down).
    def run_ppi_diff_pass(score_type):
        variant_cfg = copy.deepcopy(ppi_cfg)
        variant_cfg['score_type'] = score_type
        variant_yaml = os.path.join(os.path.dirname(YAML_BY_MODE['ppi']), f'_ppi_diff_{score_type}.yaml')
        with open(variant_yaml, 'w') as f:
            yaml.safe_dump(variant_cfg, f)
        run_be3d(variant_yaml)

    run_ppi_diff_pass('LFC3D')

    def show_ppi_qa(gene, screen_name):
        noppi_dir = os.path.join(ppi_root, 'no_ppi', gene)
        ppi_dir = os.path.join(ppi_root, 'ppi', gene)

        print(f'{gene} -- QA (KS2 test), no-PPI then PPI-mode:')
        show_side_by_side(
            plot_hypothesis_qa(noppi_dir, test='KolmogorovSmirnov'),
            plot_hypothesis_qa(ppi_dir, test='KolmogorovSmirnov'),
            width=600, height=400,
        )
        print(f'{gene} -- QA (MW test), no-PPI then PPI-mode:')
        show_side_by_side(
            plot_hypothesis_qa(noppi_dir, test='MannWhitney'),
            plot_hypothesis_qa(ppi_dir, test='MannWhitney'),
            width=600, height=400,
        )
        print('Processed LFC distribution by mutation category (violin, post mutation_priority + '
                'per-category filtering), no-PPI then PPI-mode:')
        show_side_by_side(
            plot_violin_by_processed_muttype(noppi_dir, gene, screen_name),
            plot_violin_by_processed_muttype(ppi_dir, gene, screen_name),
            width=600, height=400,
        )

    interact(
        show_ppi_qa,
        gene=Dropdown(options=gene_names, description='Gene:'),
        screen_name=Dropdown(options=ppi_screens, description='Screen:'),
    );

else:
    blind_cfg = load_yaml(YAML_BY_MODE['blind_target'])
    blind_dir = blind_cfg['output_dir']
    blind_gene, blind_chain = blind_cfg['input_gene'], blind_cfg['input_chain']
    blind_partners = blind_cfg['partners']
    blind_tsv = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D.tsv')

    # blind_target's target itself never gets hypothesis_test/screendata (run_blind_target skips
    # both -- it has no screen data of its own). Each partner runs through parse_be_data +
    # prioritize_by_sequence (preprocess_ppi_partner), same as a monomer run, but
    # preprocess_ppi_partner explicitly skips hypothesis_test -- so there's no KS2/MW QA to show
    # even for the partner, just its processed-LFC violin, under
    # {blind_dir}/ppi_partners/{gene}_chain_{chain}/screendata/.
    if not os.path.exists(blind_tsv):
        run_be3d(YAML_BY_MODE['blind_target'])

    partner_by_gene = {p['gene']: p for p in blind_partners}
    partner_names = list(partner_by_gene)
    partner_screens_by_gene = {
        gene: [s.strip().split('.')[0] for s in p['screens'].split(',')]
        for gene, p in partner_by_gene.items()
    }
    all_partner_screens = sorted({s for screens in partner_screens_by_gene.values() for s in screens})

    print("Note: blind_target partners skip hypothesis_test (preprocess_ppi_partner), so no KS2/MW "
            "QA plot is available -- showing each partner's processed LFC distribution (violin) instead:")

    def show_blind_qa(partner_gene, screen_name):
        if screen_name not in partner_screens_by_gene[partner_gene]:
            print(f"{partner_gene} has no screen '{screen_name}' -- pick one of {partner_screens_by_gene[partner_gene]}")
            return
        partner_chain = partner_by_gene[partner_gene]['chain']
        partner_dir = os.path.join(blind_dir, 'ppi_partners', f'{partner_gene}_chain_{partner_chain}')

        violin_fig = plot_violin_by_processed_muttype(partner_dir, partner_gene, screen_name)
        if violin_fig is not None:
            display(violin_fig)

    interact(
        show_blind_qa,
        partner_gene=Dropdown(options=partner_names, description='Partner:'),
        screen_name=Dropdown(options=all_partner_screens, description='Screen:'),
    );


[ppi_diff] skipping PPI leg, already completed in /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1/ppi
[ppi_diff] skipping no-PPI leg for KBTBD4, already completed in /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1/no_ppi/KBTBD4
[ppi_diff] skipping no-PPI leg for HDAC1, already completed in /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1/no_ppi/HDAC1
wrote /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1/ppi_vs_noppi_abe_neg_control.tsv (1016 residues across 2 chains)
wrote noppi_abe_neg_control.pdb, ppi_abe_neg_control.pdb, delta_abe_neg_control.pdb in /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1
wrote /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1/ppi_vs_noppi_cbe_neg_control.tsv (1016 residues across 2 chains)
wrote noppi_cbe_neg_control.pdb, ppi_cbe_neg_control.pdb, delta_cbe_neg_control.pdb in /Users/ymyung/Projects/BEClust3D/be3d_test/o

interactive(children=(Dropdown(description='Gene:', options=('KBTBD4', 'HDAC1'), value='KBTBD4'), Dropdown(des…

# Monomer Mode

## BE-Clust3D
- Residue dot-plot for single screens (LFC and LFC3D)
- Scatter plots of LFC and LFC3D comparison for single screens
- Dendrogram of LFC and LFC3D

In [ ]:
if MODE == 'monomer':
    def show_monomer_clust3d(screen_name):
        print('Residue dot-plots, LFC (positive, negative):')
        show_side_by_side(
            plot_score_scatter(monomer_dir, monomer_gene, screen_name, score_type='LFC', direction='positive'),
            plot_score_scatter(monomer_dir, monomer_gene, screen_name, score_type='LFC', direction='negative'),
    		width=600, height=400
        )
        print('Residue dot-plots, LFC3D (positive, negative):')
        show_side_by_side(
            plot_score_scatter(monomer_dir, monomer_gene, screen_name, score_type='LFC3D', direction='positive'),
            plot_score_scatter(monomer_dir, monomer_gene, screen_name, score_type='LFC3D', direction='negative'),
    		width=600, height=400
        )

        print('LFC vs. LFC3D (residues with LFC3D but no LFC shown in the left strip):')
        fig = plot_lfc_lfc3d_scatter(monomer_dir, monomer_gene, screen_name, width=500, height=400)
        if fig is not None:
            display(fig)

        print('pLDDT vs. RSA / LFC3D hit count by domain / pLDDT-disorder category:')
        show_side_by_side(
            plot_plddt_rsa_scatter(monomer_dir, monomer_gene, screen_name),
            plot_domain_barplot(monomer_dir, monomer_gene, monomer_uniprot, screen_name),
            plot_plddt_dis_barplot(monomer_dir, monomer_gene, screen_name),
            height=400, width=600
        )

        print('Enrichment test (pLDDT-disorder, log2 odds ratio):')
        fig = plot_enrichment_test(monomer_dir, monomer_gene, screen_name=screen_name, width=400, height=300)
        if fig is not None:
            display(fig)
            
        print('Dendrogram (p<0.05) -- pick a score type / direction:')
        show_picker({
            'LFC positive': plot_dendrogram(monomer_dir, monomer_gene, screen_name, score_type='LFC', direction='Positive', height=400),
            'LFC negative': plot_dendrogram(monomer_dir, monomer_gene, screen_name, score_type='LFC', direction='Negative', height=400),
            'LFC3D positive': plot_dendrogram(monomer_dir, monomer_gene, screen_name, score_type='LFC3D', direction='Positive', height=400),
            'LFC3D negative': plot_dendrogram(monomer_dir, monomer_gene, screen_name, score_type='LFC3D', direction='Negative', height=400),
        }, description='Dendrogram:')

    interact(show_monomer_clust3d, screen_name=Dropdown(options=monomer_screens, description='Screen:'));
else:
    print(f"[skipped] mode is '{MODE}', not 'monomer' -- skipping BE-Clust3D (monomer).")


## BE-MetaClust3D
- If multiple screens
- Residue dot-plot for single screens (meta-LFC and meta-LFC3D)
- Scatter plots of meta-LFC and meta-LFC3D comparison for single screens
- Dendrogram of meta-LFC and -metaLFC3D

In [ ]:
if MODE == 'monomer':
    monomer_func_meta = monomer_cfg['function_for_meta']

    if len(monomer_screens) > 1:
        print('Meta residue dot-plots, meta-LFC (positive, negative):')
        show_side_by_side(
            plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC', direction='positive'),
            plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC', direction='negative'),
    		width=600, height=400
        )
        print('Meta residue dot-plots, meta-LFC3D (positive, negative):')
        show_side_by_side(
            plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC3D', direction='positive'),
            plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC3D', direction='negative'),
    		width=600, height=400
        )

        print('meta-LFC vs. meta-LFC3D (residues with meta-LFC3D but no meta-LFC shown in the left strip):')
        fig = plot_meta_lfc_lfc3d_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, width=500, height=400)
        if fig is not None:
            display(fig)

        print('pLDDT vs. RSA / Meta LFC3D hit count by domain / pLDDT-disorder category / Meta enrichment test (pLDDT-disorder, log2 odds ratio):')
        show_side_by_side(
            plot_meta_plddt_rsa_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta),
            plot_meta_domain_barplot(monomer_dir, monomer_gene, monomer_uniprot, function_for_meta=monomer_func_meta),
            plot_meta_plddt_dis_barplot(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta),
            width=600, height=400
        )

        print('Enrichment test (pLDDT-disorder, log2 odds ratio):')
        fig = plot_enrichment_test(monomer_dir, monomer_gene, screen_name=None, width=400, height=300)
        if fig is not None:
            display(fig)
            
        print('Meta dendrogram (p<0.05) -- pick a score type / direction:')
        show_picker({
            'Meta LFC positive': plot_meta_dendrogram(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC', direction='Positive', height=400),
            'Meta LFC negative': plot_meta_dendrogram(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC', direction='Negative', height=400),
            'Meta LFC3D positive': plot_meta_dendrogram(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC3D', direction='Positive', height=400),
            'Meta LFC3D negative': plot_meta_dendrogram(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC3D', direction='Negative', height=400),
        }, description='Dendrogram:')
    else:
        print('Only one screen -- no meta-aggregation to show.')
else:
    print(f"[skipped] mode is '{MODE}', not 'monomer' -- skipping BE-MetaClust3D (monomer).")


# PPI mode

## BE-Clust3D
- Residue dot-plot for single screens (LFC and LFC3D) and PPI mode LFC3D
- Scatter plots of single screen LFC3Ds (x-axis) vs PPI mode LFC3D (y-axis)
- Dendrogram of single screen  LFC and LFC3D and PPI mode LFC3D

In [10]:
if MODE == 'ppi':
    ppi_cfg = load_yaml(PPI_YAML)
    ppi_root = ppi_cfg['output_dir']
    gene_names = [g.strip() for g in ppi_cfg['input_gene'].split(',')]
    chain_list = [c.strip() for c in ppi_cfg['input_chain'].split(',')]
    ppi_screens = [s.strip().split('.')[0] for s in ppi_cfg['screens'].split(',')]

    # mode: ppi_diff runs the PPI leg (mode: complex) and no-PPI leg (mode: monomer, per gene)
    # once, then merges -- skip_existing makes each pass a no-op for the pipeline legs once the
    # first pass has run them. score_type controls only the (cheap) merge step: 'LFC3D' produces
    # one merged TSV+PDB set per screen; 'Meta_LFC3D' produces the meta-aggregated one (needed by
    # the BE-MetaClust3D and Merged-results sections below).
    def run_ppi_diff_pass(score_type):
        variant_cfg = copy.deepcopy(ppi_cfg)
        variant_cfg['score_type'] = score_type
        variant_yaml = os.path.join(os.path.dirname(PPI_YAML), f'_ppi_diff_{score_type}.yaml')
        with open(variant_yaml, 'w') as f:
            yaml.safe_dump(variant_cfg, f)
        run_be3d(variant_yaml)

    run_ppi_diff_pass('LFC3D')
    run_ppi_diff_pass('Meta_LFC3D')

    ppi_func_meta = ppi_cfg['function_for_meta']
    ppi_uniprot_by_gene = dict(zip(gene_names, [u.strip() for u in ppi_cfg['input_uniprot'].split(',')]))

    def show_ppi_clust3d(gene, screen_name):
        chain = dict(zip(gene_names, chain_list))[gene]
        uniprot = ppi_uniprot_by_gene[gene]
        noppi_dir = os.path.join(ppi_root, 'no_ppi', gene)
        ppi_dir = os.path.join(ppi_root, 'ppi', gene)

        print(f'{gene} (chain {chain}) -- residue dot-plots, LFC3D positive -- no-PPI, then PPI-mode:')
        show_side_by_side(
            plot_score_scatter(noppi_dir, gene, screen_name, score_type='LFC3D', direction='positive'),
            plot_score_scatter(ppi_dir, gene, screen_name, score_type='LFC3D', direction='positive'),
            width=600, height=400
        )
        print('residue dot-plots, LFC3D negative -- no-PPI, then PPI-mode:')
        show_side_by_side(
            plot_score_scatter(noppi_dir, gene, screen_name, score_type='LFC3D', direction='negative'),
            plot_score_scatter(ppi_dir, gene, screen_name, score_type='LFC3D', direction='negative'),
            width=600, height=400
        )

        print('no-PPI LFC3D (x) vs. PPI-mode LFC3D (y):')
        df_screen = pd.read_csv(os.path.join(ppi_root, f'ppi_vs_noppi_{screen_name}.tsv'), sep='\t')
        plot_ppi_vs_noppi_scatter(df_screen[df_screen['gene'] == gene], 'LFC3D', width=500, height=400)

        print('LFC vs. LFC3D -- no-PPI, then PPI-mode (residues with LFC3D but no LFC shown in each left strip):')
        show_side_by_side(
            plot_lfc_lfc3d_scatter(noppi_dir, gene, screen_name),
            plot_lfc_lfc3d_scatter(ppi_dir, gene, screen_name),
            width=500, height=500
        )

        print('pLDDT vs. RSA -- no-PPI, then PPI-mode:')
        show_side_by_side(
            plot_plddt_rsa_scatter(noppi_dir, gene, screen_name),
            plot_plddt_rsa_scatter(ppi_dir, gene, screen_name),
            width=500, height=500
        )

        print('LFC3D hit count by pLDDT-disorder category -- no-PPI, then PPI-mode:')
        show_side_by_side(
            plot_plddt_dis_barplot(noppi_dir, gene, screen_name),
            plot_plddt_dis_barplot(ppi_dir, gene, screen_name),
            width=600, height=400
        )

        print('Enrichment test (pLDDT-disorder) -- no-PPI, then PPI-mode:')
        show_side_by_side(
            plot_enrichment_test(noppi_dir, gene, screen_name=screen_name),
            plot_enrichment_test(ppi_dir, gene, screen_name=screen_name),
            height=400, width=600
        )

        print('LFC3D dendrogram (p<0.05) -- pick a mode / direction:')
        show_picker({
            'no-PPI positive': plot_dendrogram(noppi_dir, gene, screen_name, score_type='LFC3D', direction='Positive', height=400),
            'PPI-mode positive': plot_dendrogram(ppi_dir, gene, screen_name, score_type='LFC3D', direction='Positive', height=400),
            'no-PPI negative': plot_dendrogram(noppi_dir, gene, screen_name, score_type='LFC3D', direction='Negative', height=400),
            'PPI-mode negative': plot_dendrogram(ppi_dir, gene, screen_name, score_type='LFC3D', direction='Negative', height=400),
        }, description='Dendrogram:')

    interact(
        show_ppi_clust3d,
        gene=Dropdown(options=gene_names, description='Gene:'),
        screen_name=Dropdown(options=ppi_screens, description='Screen:'),
    );
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping BE-Clust3D (ppi).")


[ppi_diff] skipping PPI leg, already completed in /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1/ppi
[ppi_diff] skipping no-PPI leg for KBTBD4, already completed in /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1/no_ppi/KBTBD4
[ppi_diff] skipping no-PPI leg for HDAC1, already completed in /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1/no_ppi/HDAC1
wrote /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1/ppi_vs_noppi_abe_neg_control.tsv (1016 residues across 2 chains)
wrote noppi_abe_neg_control.pdb, ppi_abe_neg_control.pdb, delta_abe_neg_control.pdb in /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1
wrote /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1/ppi_vs_noppi_cbe_neg_control.tsv (1016 residues across 2 chains)
wrote noppi_cbe_neg_control.pdb, ppi_cbe_neg_control.pdb, delta_cbe_neg_control.pdb in /Users/ymyung/Projects/BEClust3D/be3d_test/o

interactive(children=(Dropdown(description='Gene:', options=('KBTBD4', 'HDAC1'), value='KBTBD4'), Dropdown(des…

## BE-MetaClust3D
- If multiple screens
- Residue dot-plot for single screens (meta-LFC and meta-LFC3D) and PPI mode meta-LFC3D
- Scatter plots of meta-LFC3D (single screens, x-axis) vs meta-LFC3D (PPI mode, y-axis)
- Dendrogram of meta-LFC and meta-LFC3D (single screens) and meta-LFC3D (PPI-mode)

In [11]:
if MODE == 'ppi':
    if len(ppi_screens) > 1:
        df_meta = pd.read_csv(os.path.join(ppi_root, 'ppi_vs_noppi_Meta_LFC3D.tsv'), sep='\t')

        def show_ppi_metaclust3d(gene):
            chain = dict(zip(gene_names, chain_list))[gene]
            noppi_dir = os.path.join(ppi_root, 'no_ppi', gene)
            ppi_dir = os.path.join(ppi_root, 'ppi', gene)

            print(f'{gene} (chain {chain}) -- meta residue dot-plots, meta-LFC3D positive -- no-PPI, then PPI-mode:')
            show_side_by_side(
                plot_meta_score_scatter(noppi_dir, gene, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='positive'),
                plot_meta_score_scatter(ppi_dir, gene, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='positive'),
                width=600, height=400
            )
            print('meta residue dot-plots, meta-LFC3D negative -- no-PPI, then PPI-mode:')
            show_side_by_side(
                plot_meta_score_scatter(noppi_dir, gene, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='negative'),
                plot_meta_score_scatter(ppi_dir, gene, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='negative'),
                width=600, height=400
            )

            print('no-PPI meta-LFC3D (x) vs. PPI-mode meta-LFC3D (y):')
            plot_ppi_vs_noppi_scatter(df_meta[df_meta['gene'] == gene], 'meta-LFC3D', width=500, height=400)

            print('meta-LFC vs. meta-LFC3D -- no-PPI, then PPI-mode (residues with meta-LFC3D but no meta-LFC shown in each left strip):')
            show_side_by_side(
                plot_meta_lfc_lfc3d_scatter(noppi_dir, gene, function_for_meta=ppi_func_meta),
                plot_meta_lfc_lfc3d_scatter(ppi_dir, gene, function_for_meta=ppi_func_meta),
                width=500, height=500
            )

            print('pLDDT vs. RSA -- no-PPI, then PPI-mode:')
            show_side_by_side(
                plot_meta_plddt_rsa_scatter(noppi_dir, gene, function_for_meta=ppi_func_meta),
                plot_meta_plddt_rsa_scatter(ppi_dir, gene, function_for_meta=ppi_func_meta),
                width=600, height=400
            )

            print('Meta LFC3D hit count by pLDDT-disorder category -- no-PPI, then PPI-mode:')
            show_side_by_side(
                plot_meta_plddt_dis_barplot(noppi_dir, gene, function_for_meta=ppi_func_meta),
                plot_meta_plddt_dis_barplot(ppi_dir, gene, function_for_meta=ppi_func_meta),
                width=600, height=400
            )

            print('Meta enrichment test (pLDDT-disorder) -- no-PPI, then PPI-mode:')
            show_side_by_side(
                plot_enrichment_test(noppi_dir, gene, screen_name=None),
                plot_enrichment_test(ppi_dir, gene, screen_name=None),
                width=600, height=400
            )

            print('Meta LFC3D dendrogram (p<0.05) -- pick a mode / direction:')
            show_picker({
                'no-PPI positive': plot_meta_dendrogram(noppi_dir, gene, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='Positive', height=400),
                'PPI-mode positive': plot_meta_dendrogram(ppi_dir, gene, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='Positive', height=400),
                'no-PPI negative': plot_meta_dendrogram(noppi_dir, gene, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='Negative', height=400),
                'PPI-mode negative': plot_meta_dendrogram(ppi_dir, gene, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='Negative', height=400),
            }, description='Dendrogram:')

        interact(show_ppi_metaclust3d, gene=Dropdown(options=gene_names, description='Gene:'));
    else:
        print('Only one screen -- no meta-aggregation to show.')
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping BE-MetaClust3D (ppi).")


interactive(children=(Dropdown(description='Gene:', options=('KBTBD4', 'HDAC1'), value='KBTBD4'), Output()), _…

## Merged results (meta-LFC3D)
- Show a table and sort by highest ∆meta-LFC3D
- Show the 3D viewe of three structures (no-PPI, PPI and diff)
- - color by meta-LFC3D (-2 negative min, +2 positive max, 0 white) and |diff| > 1 residues and its Calpha as spheres.

In [12]:
if MODE == 'ppi':
    df_merged = pd.read_csv(os.path.join(ppi_root, 'ppi_vs_noppi_Meta_LFC3D.tsv'), sep='\t')
    df_merged_sorted = df_merged.reindex(df_merged['delta_score'].abs().sort_values(ascending=False).index)

    print('Top 10 residues by |delta meta-LFC3D| (PPI - no-PPI):')
    top10 = df_merged_sorted.head(10)
    display(top10[['gene', 'chain', 'unipos', 'unires', 'noppi_score', 'ppi_score', 'delta_score']])
    top10_unipos = top10['unipos'].tolist()

    base_pdb = os.path.join(ppi_root, 'ppi', gene_names[0], 'sequence_structure')
    base_pdb = os.path.join(base_pdb, [f for f in os.listdir(base_pdb) if f.endswith('_processed.pdb')][0])

    merged_views = {
        'No-PPI meta-LFC3D': chain_values_from_df(df_merged, 'noppi_score'),
        'PPI-mode meta-LFC3D': chain_values_from_df(df_merged, 'ppi_score'),
        'Delta (PPI - no-PPI) meta-LFC3D': chain_values_from_df(df_merged, 'delta_score'),
    }

    print('Structure colored by the selected view (spheres = the top 10 |delta| residues above):')
    merged_widget = PDBeMolstar(hide_water=True, height='500px')
    load_molstar_pdb(merged_widget, base_pdb)
    display(merged_widget)

    def show_merged_view(view_name):
        color_molstar(merged_widget, merged_views[view_name], vmax=2.0, highlight_top_n=10)

    interact(show_merged_view, view_name=Dropdown(options=list(merged_views), description='View:'));
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping PPI merged results.")


Top 10 residues by |delta meta-LFC3D| (PPI - no-PPI):


,gene,chain,unipos,unires,noppi_score,ppi_score,delta_score
330,KBTBD4,B,331,P,-0.055402,2.265313,2.320715
331,KBTBD4,B,332,R,0.262081,2.140287,1.878206
332,KBTBD4,B,333,D,0.394308,2.272514,1.878206
31,KBTBD4,B,32,F,1.290539,-0.247093,-1.537632
33,KBTBD4,B,34,N,1.647224,1.030698,-0.616526
121,KBTBD4,B,122,G,0.466410,0.921791,0.455381
125,KBTBD4,B,126,L,-0.241375,0.169219,0.410594
511,KBTBD4,B,512,A,-0.583967,-0.199322,0.384645
26,KBTBD4,B,27,M,-0.414375,-0.047508,0.366868
25,KBTBD4,B,26,S,-0.414375,-0.047508,0.366868


Structure colored by the selected view (spheres = the top 10 |delta| residues above):


interactive(children=(Dropdown(description='View:', options=('No-PPI meta-LFC3D', 'PPI-mode meta-LFC3D', 'Delt…

# Blind target mode
- in a table, highlight those have LFC3D values (single and meta)
- Residue-dot plot for the above table (one screen, then use than, if multiple screens, then use meta-LFC3D)
- Use Molstar viewer to visualize the new LFC3D value in 3D structure. Blue for + and Neg for - LFC3D  (one screen, then use than, if multiple screens, then use meta-LFC3D)

In [ ]:
if MODE == 'blind_target':
    blind_cfg = load_yaml(YAML_BY_MODE['blind_target'])
    blind_dir = blind_cfg['output_dir']
    blind_gene, blind_chain = blind_cfg['input_gene'], blind_cfg['input_chain']
    blind_tsv = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D.tsv')

    if not os.path.exists(blind_tsv):
        run_be3d(YAML_BY_MODE['blind_target'])

    df_blind = pd.read_csv(blind_tsv, sep='\t')

    # use the meta-aggregated column if there's more than one partner screen, else the single screen's
    screen_names = [c[:-len('_LFC3D_blind_overall')] for c in df_blind.columns if c.endswith('_LFC3D_blind_overall')]
    if 'Meta_LFC3D_blind_overall' in df_blind.columns:
        neg_col, pos_col, overall_col = 'Meta_LFC3D_blind_neg', 'Meta_LFC3D_blind_pos', 'Meta_LFC3D_blind_overall'
    else:
        screen_name = screen_names[0]
        neg_col, pos_col, overall_col = f'{screen_name}_LFC3D_blind_neg', f'{screen_name}_LFC3D_blind_pos', f'{screen_name}_LFC3D_blind_overall'

    df_blind_hits = df_blind[(df_blind[neg_col] != '-') | (df_blind[pos_col] != '-')]
    print(f'{blind_gene} (chain {blind_chain}) -- {len(df_blind_hits)}/{len(df_blind)} residues have a blind LFC3D value:')
    display(df_blind_hits[['unipos', 'unires', 'chain', neg_col, pos_col, overall_col]])

    print('Residue dot-plot (signed value: negative or positive column, whichever is set):')
    signed = pd.to_numeric(df_blind[neg_col].replace('-', pd.NA), errors='coerce')
    signed = signed.fillna(pd.to_numeric(df_blind[pos_col].replace('-', pd.NA), errors='coerce'))
    df_blind_signed = df_blind.copy()
    df_blind_signed['_signed_blind_LFC3D'] = signed
    plot_residue_dot(df_blind_signed, '_signed_blind_LFC3D', f'{blind_gene} blind LFC3D')

    print('3D structure colored by the selected view (blue = positive, red = negative):')
    overall_pdb = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D_overall.pdb')
    pos_pdb = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D_pos.pdb')
    neg_pdb = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D_neg.pdb')
    # run_blind_target always writes all three PDBs together (same coordinates, different
    # B-factors baked in) when user_pdb is set, so which one is loaded as the base structure
    # doesn't matter -- only the color_data changes per dropdown selection below.
    blind_base_pdb = overall_pdb if os.path.exists(overall_pdb) else (pos_pdb if os.path.exists(pos_pdb) else neg_pdb)

    def _blind_chain_values(col):
        values = pd.to_numeric(df_blind[col].replace('-', pd.NA), errors='coerce')
        return {blind_chain: {int(p): float(v) for p, v in zip(df_blind['unipos'], values) if pd.notna(v)}}

    blind_views = {
        'Negative': _blind_chain_values(neg_col),
        'Positive': _blind_chain_values(pos_col),
        'Overall': _blind_chain_values(overall_col),
    }

    blind_widget = PDBeMolstar(hide_water=True, height='500px')
    load_molstar_pdb(blind_widget, blind_base_pdb)
    display(blind_widget)

    def show_blind_view(view_name):
        color_molstar(blind_widget, blind_views[view_name], vmax=2.0, highlight_top_n=10)

    interact(show_blind_view, view_name=Dropdown(options=list(blind_views), description='View:'));
else:
    print(f"[skipped] mode is '{MODE}', not 'blind_target' -- skipping blind-target results.")
